# Deliverable 1 · Step 1 — Export your EE541 SimpleCNN to ONNX (and prove it's faithful)

**Goal (small on purpose):** turn your *real* best model (`SimpleCNN` + Mixup, the `E3_mixup` checkpoint) into a
portable **ONNX** graph that NVIDIA Triton can serve — then *verify* ONNX gives the same answers as PyTorch.
We do NOT touch Triton yet. One concept per step.

**You'll learn:** (1) what ONNX is and why serving stacks use it; (2) why the batch axis must be **dynamic**;
(3) how to **prove** an export is faithful (don't trust — verify).

Run top-to-bottom on a **Colab T4** (Runtime → Change runtime type → T4 GPU).


## 0. Environment — confirm GPU + install ONNX tooling
Torch ships with Colab. We add `onnx` (the format) and `onnxruntime-gpu` (to run + verify the ONNX).


In [ ]:
# Step 1 runs on CPU (export + verify need no GPU). onnxruntime-gpu's newest build wants CUDA 13,
# which Colab may not have -> libcudart error. Plain onnxruntime avoids the whole version dance.
!nvidia-smi -L    # just to confirm a T4 is attached (used later in Step 3)
!pip -q uninstall -y onnxruntime onnxruntime-gpu 2>/dev/null
!pip -q install onnx onnxruntime
import torch, onnx, onnxruntime as ort, numpy as np
print('torch', torch.__version__, '| ort', ort.__version__)


## 1. Upload your trained checkpoint
No GitHub token needed — just upload **one** fold's checkpoint (e.g. `E3_mixup_fold1_best.pt`). It's a small file
(this model is tiny). On your Mac it's at:
`~/Documents/USC/Spring-26/EE 541/Project/urban-sound-clf/results/checkpoints/E3_mixup_fold1_best.pt`


In [ ]:
from google.colab import files
up = files.upload()                      # pick your E3_mixup_fold*_best.pt
CKPT = next(iter(up))                     # the uploaded filename
print('uploaded:', CKPT)


## 2. Your model, exactly as shipped
This is your real `SimpleCNN` (pasted verbatim from `src/models/simple_cnn.py`) so the checkpoint loads with no
key mismatch. Note `forward()` returns **only logits** — the 256-D embedding lives in `get_penultimate()` and is
**not** served. A server exposes a narrow interface; the research model is wider. (Good interview point.)


In [ ]:
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, pool=True):
        super().__init__()
        layers = [nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
                  nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)]
        if pool: layers.append(nn.MaxPool2d(2, 2))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.5):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1, 32), ConvBlock(32, 64), ConvBlock(64, 128, pool=False),
            nn.AdaptiveAvgPool2d((4, 4)))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128*4*4, 256), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(256, num_classes))
    def forward(self, x): return self.classifier(self.features(x))


## 3. Load the trained weights
Instantiate and load the checkpoint. We handle both `{'model_state_dict': ...}` and raw-state-dict formats.
Empty `missing`/`unexpected` lists = perfect load.


In [ ]:
model = SimpleCNN(num_classes=10)
state = torch.load(CKPT, map_location='cpu', weights_only=False)  # your own file -> trusted
sd = state.get('model_state_dict', state) if isinstance(state, dict) else state
missing, unexpected = model.load_state_dict(sd, strict=False)
print('missing:', list(missing))
print('unexpected:', list(unexpected))
model.eval()

dummy = torch.randn(1, 1, 128, 128)          # (batch=1, 1 mel channel, 128 mel bins, 128 frames)
with torch.no_grad(): print('logits shape:', tuple(model(dummy).shape))   # expect (1, 10)


## 4. Export to ONNX — with a **dynamic batch axis**
`torch.onnx.export` traces the model into a portable ONNX graph. The key arg is `dynamic_axes`: we mark **axis 0**
(batch) of input and output as variable. Without it the graph is frozen at batch=1 and Triton could never fuse
requests into a batch. (`dynamo=False` = the reliable classic exporter; `dynamo=True` is the newer torch.export
path — same idea, worth knowing the name.)


In [ ]:
ONNX_PATH = 'audio_cnn.onnx'
torch.onnx.export(
    model, dummy, ONNX_PATH,
    input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17, dynamo=False)
onnx.checker.check_model(onnx.load(ONNX_PATH))
print('exported + structurally valid:', ONNX_PATH)


## 5. **Verify faithfulness** — prove ONNX == PyTorch
Exports can silently diverge. Run the ONNX in onnxruntime and compare to PyTorch at **two batch sizes** (this also
proves the dynamic axis works). Max abs diff < 1e-4 = faithful. **Put this number in your README** — it's exactly
the kind of rigor NVIDIA looks for.


In [ ]:
sess = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
for n in (1, 8):
    x = torch.randn(n, 1, 128, 128)
    with torch.no_grad(): torch_logits = model(x).numpy()
    onnx_logits = sess.run(['logits'], {'input': x.numpy()})[0]
    assert onnx_logits.shape == (n, 10), onnx_logits.shape
    d = float(np.abs(torch_logits - onnx_logits).max())
    print(f'N={n}: shapes match, max|torch-onnx| = {d:.2e}', ' OK' if d < 1e-4 else ' <-- TOO HIGH')
print('\nRecord the max diff in your README.')


## 6. Save the ONNX for Step 2
Download it — in Step 2 it goes into Triton at `model_repository/audio_cnn/1/model.onnx`.


In [ ]:
from google.colab import files
files.download(ONNX_PATH)


## Recap + self-check
- **ONNX** = portable model graph; the serving on-ramp (and doorway to TensorRT).
- **Dynamic batch axis** = what makes server-side batching possible.
- **Verify numerically** — you now have a measured faithfulness number.

**Say out loud:** (1) What breaks if axis 0 is fixed at 1? (2) Why serve logits, not the embedding? (3) How would
you catch a bad export *before* production?

**Next — Step 2:** load `audio_cnn.onnx` into Triton (authentic Docker container on your Mac, real `config.pbtxt`),
send it a request, then Step 3: benchmark CPU (Mac) vs GPU (T4).
